# 06 — Grad-CAM Interpretability

Apply Gradient-weighted Class Activation Mapping (Grad-CAM) to a trained model
to visualize which spatial regions within each 244×244 AlphaEarth embedding tile
most influence the model's coverage classification decision.

Inputs:
- Trained model checkpoint (`.ckpt`)
- `.npz` embedding tiles

Outputs:
- Grad-CAM heatmaps overlaid on embedding RGB visualizations

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import glob
import os
from pathlib import Path

from model.pipeline.models import ViTClassifier, CNNClassifier, Classifier
from model.pipeline.modules import LitModule
from model.pipeline.gradcam import GradCAM

## 1. Configuration

In [ ]:
# ---- USER CONFIG ----
CHECKPOINT_PATH = '../../output/checkpoints/best_model.ckpt'
EMBEDDING_DIR   = '../../output/dataset/'
DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'

MODEL_CLASS = 'ViTClassifier'  # 'CNNClassifier' or 'Classifier'
NUM_SAMPLES = 8                # Number of tiles to visualize

CATEGORY_NAMES = ['1-20%', '21-40%', '41-60%', '61-80%', '81-100%']

print(f'Device: {DEVICE}')

## 2. Load Model & Initialize Grad-CAM

In [ ]:
# Instantiate architecture
model_classes = {
    'ViTClassifier': ViTClassifier,
    'CNNClassifier': CNNClassifier,
    'Classifier': Classifier,
}

net = model_classes[MODEL_CLASS](in_channels=64, out_features=5)

# Load checkpoint
if os.path.exists(CHECKPOINT_PATH):
    lit = LitModule.load_from_checkpoint(CHECKPOINT_PATH, net=net)
    model = lit.net
    print(f'Loaded checkpoint from {CHECKPOINT_PATH}')
else:
    model = net
    print(f'WARNING: no checkpoint found — using untrained model for demo.')

model = model.to(DEVICE).eval()

# Select target layer for Grad-CAM depending on architecture
if MODEL_CLASS == 'ViTClassifier':
    target_layer = model.blocks[-1].norm1   # last transformer block
elif MODEL_CLASS == 'CNNClassifier':
    target_layer = model.features[-3]       # last conv layer before pool
else:
    target_layer = model.net[0]              # AdaptiveAvgPool

gradcam = GradCAM(model, target_layer)
print(f'Grad-CAM target layer: {target_layer.__class__.__name__}')

## 3. Helper: Pseudo-RGB from AlphaEarth Embeddings

Since embeddings have 64 bands, we create a false-colour composite
using the first 3 principal components (or simply bands 0, 1, 2)
to provide spatial context alongside the Grad-CAM heatmap.

In [ ]:
def embeddings_to_pseudo_rgb(emb: np.ndarray) -> np.ndarray:
    """
    Convert a (64, H, W) embedding to a (H, W, 3) pseudo-RGB image.
    Uses bands 0, 1, 2 with percentile-based contrast stretching.
    """
    rgb = emb[:3].transpose(1, 2, 0).copy()  # (H, W, 3)
    for c in range(3):
        lo, hi = np.percentile(rgb[:, :, c], [2, 98])
        rgb[:, :, c] = np.clip((rgb[:, :, c] - lo) / (hi - lo + 1e-8), 0, 1)
    return rgb


def overlay_heatmap(rgb: np.ndarray, heatmap: np.ndarray,
                    alpha: float = 0.5) -> np.ndarray:
    """
    Overlay a Grad-CAM heatmap on a pseudo-RGB image.
    """
    cmap = plt.cm.jet
    heatmap_color = cmap(heatmap)[:, :, :3]  # (H, W, 3)
    blended = (1 - alpha) * rgb + alpha * heatmap_color
    return np.clip(blended, 0, 1)

## 4. Generate Grad-CAM Heatmaps

In [ ]:
npz_files = sorted(glob.glob(os.path.join(EMBEDDING_DIR, '*.npz')))[:NUM_SAMPLES]
print(f'Generating Grad-CAM for {len(npz_files)} tiles...\n')

fig, axes = plt.subplots(len(npz_files), 3, figsize=(15, 5 * len(npz_files)))
if len(npz_files) == 1:
    axes = axes[np.newaxis, :]

for i, fpath in enumerate(npz_files):
    # Load embedding
    data = np.load(fpath)
    key = 'embeddings' if 'embeddings' in data else list(data.keys())[0]
    emb_np = data[key].astype(np.float32)  # (64, H, W)
    emb_tensor = torch.from_numpy(emb_np).unsqueeze(0).to(DEVICE)  # (1, 64, H, W)

    # Predict
    with torch.no_grad():
        logits = model(emb_tensor)
        probs = torch.softmax(logits, dim=1)[0]
        pred_class = probs.argmax().item()
        confidence = probs[pred_class].item()

    # Grad-CAM
    heatmap = gradcam(emb_tensor, target_class=pred_class,
                      img_size=(emb_np.shape[1], emb_np.shape[2]))

    # Visualize
    rgb = embeddings_to_pseudo_rgb(emb_np)
    overlay = overlay_heatmap(rgb, heatmap, alpha=0.5)

    axes[i, 0].imshow(rgb)
    axes[i, 0].set_title(f'Pseudo-RGB — {Path(fpath).stem}')
    axes[i, 0].axis('off')

    im = axes[i, 1].imshow(heatmap, cmap='jet', vmin=0, vmax=1)
    axes[i, 1].set_title(f'Grad-CAM — Pred: {CATEGORY_NAMES[pred_class]} ({confidence:.1%})')
    axes[i, 1].axis('off')
    plt.colorbar(im, ax=axes[i, 1], fraction=0.046)

    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title('Overlay')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## 5. Per-Class Grad-CAM Analysis

For a single tile, generate Grad-CAM heatmaps for every class
to understand what spatial patterns the model associates with each coverage level.

In [ ]:
# Pick a single tile for per-class analysis
sample_file = npz_files[0]
data = np.load(sample_file)
key = 'embeddings' if 'embeddings' in data else list(data.keys())[0]
emb_np = data[key].astype(np.float32)
emb_tensor = torch.from_numpy(emb_np).unsqueeze(0).to(DEVICE)
rgb = embeddings_to_pseudo_rgb(emb_np)

num_classes = len(CATEGORY_NAMES)
fig, axes = plt.subplots(1, num_classes + 1, figsize=(4 * (num_classes + 1), 4))

axes[0].imshow(rgb)
axes[0].set_title('Pseudo-RGB')
axes[0].axis('off')

for c in range(num_classes):
    heatmap = gradcam(emb_tensor, target_class=c,
                      img_size=(emb_np.shape[1], emb_np.shape[2]))
    overlay = overlay_heatmap(rgb, heatmap, alpha=0.5)
    axes[c + 1].imshow(overlay)
    axes[c + 1].set_title(f'Class {CATEGORY_NAMES[c]}')
    axes[c + 1].axis('off')

plt.suptitle(f'Per-class Grad-CAM — {Path(sample_file).stem}', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Ecological Interpretation

**What to look for in the Grad-CAM heatmaps:**

- Water–vegetation boundaries: high activation along coastlines and tidal channels
  suggests the model correctly uses transition zones as discriminative features.
- Canopy density gradients: smooth activation gradients within dense mangrove
  areas indicate the model captures vegetation structure.
- Tidal channel morphology: if the model highlights linear water features within
  mangrove patches, it learns to recognize the characteristic spatial patterns of
  mangrove ecosystems.
- False patterns: if activation concentrates in corners or uniform regions,
  this may indicate overfitting.

Combined with **Monte Carlo dropout** (future work), Grad-CAM provides a complete
diagnostic toolkit: Grad-CAM reveals _what_ the model looks at, while MC dropout
reveals _how confident_ it is.

In [ ]:
# Cleanup hooks
gradcam.remove_hooks()
print('Done. Grad-CAM hooks removed.')